# Module 1 — Part 1: RAG Pipeline

Lessons 3–7: What is RAG, dataset, search, prompt, LLM.

**Provider note:** Lessons use OpenAI `responses.create`. We use Groq `chat.completions.create` — OpenAI-compatible, free tier, fast.

## Setup

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

from openai import OpenAI

# Groq is OpenAI-compatible — same client, different base_url
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-120b"

## Lesson 3 — What is RAG

First: call LLM directly with no context. It gives generic answers for course-specific questions.

In [2]:
def llm(prompt, model=MODEL):
    # Lesson uses: openai_client.responses.create(model=..., input=prompt)
    # Groq uses chat.completions with messages list
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

llm("Hey, what's up?")

"Hey there! I'm doing great—ready to help with whatever you need. What's on your mind today?"

In [3]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)
# Generic answer — LLM has no idea about our specific Zoomcamp courses

Absolutely! 🎉 If you’ve just discovered the course and are ready to dive in, you can enroll right away—provided the enrollment window is still open. Here’s a quick rundown of what you need to know and do:

---

## 1️⃣ Check the Enrollment Status
- **Open Enrollment:** Most of our courses run on a rolling‑basis, so you can join at any time.
- **Closed / Full:** If the current cohort is full, you can add your name to the waiting list and we’ll notify you as soon as a spot opens.

> **Tip:** Look for a bright “Enroll Now” button on the course homepage. If it’s grayed out, the class is at capacity.

---

## 2️⃣ Meet the Prerequisites (if any)
| Prerequisite | Why It Matters | How to Verify |
|--------------|----------------|---------------|
| Basic computer literacy | You’ll need to navigate the LMS, submit assignments, and participate in forums. | A quick self‑assessment quiz is available on the enrollment page. |
| Introductory knowledge of [topic] | Some concepts build on foundational i

### Manual context injection

Paste FAQ entries directly into the prompt. This is naive RAG — but it shows the concept.

In [4]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)
# Now the answer is correct and grounded in our FAQ data — this is RAG

Yes—you can still join the course. If you’d like to earn a certificate, just make sure to submit your project before the submission deadline.


## Lesson 4 — The Course FAQ Dataset

Fetch all FAQ documents from DataTalks.Club. These are our knowledge base.

In [5]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

print(f"Found {len(courses_raw)} courses")
courses_raw[:2]

Found 7 courses


[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 398},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [6]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)

print(f"Total documents: {len(documents)}")

Total documents: 1489


In [7]:
# Each document has: id, course, section, question, answer
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

## Lesson 5 — Search

Build a search index with minsearch. Text fields are tokenized + ranked. Keyword fields are exact-match filters.

In [8]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],  # tokenized, ranked
    keyword_fields=["course"]                         # exact filter, like SQL WHERE
)

index.fit(documents)
print("Index built")

Index built


In [9]:
# boost_dict: question matches count 2x, section matches count 0.5x
# filter_dict: only return results from llm-zoomcamp
search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

In [10]:
def search(question, course="llm-zoomcamp"):
    return index.search(
        question,
        boost_dict={"question": 2.0, "section": 0.5},
        filter_dict={"course": course},
        num_results=5
    )

search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

## Lesson 6 — Building the Prompt

Format search results into a context string, then combine with the question into a user prompt.

In [11]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
""".strip()

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [12]:
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    return USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    ).strip()

prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

## Lesson 7 — The LLM + Full RAG Pipeline

Wire search + prompt + LLM together. Use message history to separate instructions from user prompt.

In [13]:
def llm(instructions, user_prompt, model=MODEL):
    # Lesson uses responses.create with 'developer' role.
    # Groq chat.completions uses 'system' role — functionally identical.
    message_history = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = groq_client.chat.completions.create(
        model=model,
        messages=message_history
    )
    return response.choices[0].message.content

# Check token usage
raw_response = groq_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt}
    ]
)
print("Usage:", raw_response.usage)
print(raw_response.choices[0].message.content)

Usage: CompletionUsage(completion_tokens=120, prompt_tokens=621, total_tokens=741, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=72, rejected_prediction_tokens=None, text_tokens=None), prompt_tokens_details=None, queue_time=0.207147391, prompt_time=0.035038459, completion_time=0.251364512, total_time=0.286402971)
Yes—you can still join the course. Just keep in mind that if you’d like to earn a certificate, you’ll need to submit your capstone project while the program is still accepting submissions.


In [14]:
def rag(query, model=MODEL):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

# Full RAG pipeline in one call
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes—you can still join the course. Just keep in mind that if you’d like to earn a certificate, you’ll need to submit your capstone project while the course is still accepting submissions.


In [15]:
# Try more questions
print(rag("How do I get a certificate?"))

To earn a certificate you need to:

1. **Participate in a live cohort** – certificates are only issued to students who finish the course as part of a live cohort. Self‑paced study is fine for learning the material, but the final project submission and peer‑review must happen while a live cohort is open.

2. **Pass the Capstone project** – submit a project that meets the grading criteria. Homework is optional; it helps you practice but isn’t required for the certificate.

3. **Complete your assigned peer reviews** – after you submit the project, you’ll automatically receive a set of peer‑review assignments on the course platform. Finish those reviews within the given review window.

4. **Wait for the cohort’s certificate batch** – certificates are generated manually once the last project attempt and its peer‑review period are closed for the cohort. You don’t need to request it; just make sure the name shown in your profile on the course platform is exactly how you want it printed.

5. *

In [16]:
print(rag("Is it okay if I submit my homework a day late?"))

I don't know.
